Moment is encoder only. Can predict if a section of transaction history is in a certain class. This would require each pass to train a single category. As I want to train y continuously for a transaction feed, this will mean processing a transaction window for every y I train.

This seems innefficient, as with a causal mask I can train many ys in a single pass. Will reject this for now to use a decoder.

In [ ]:
from momentfm import MOMENTPipeline
import tqdm
import torch

context_length = 512
predictors = 20
num_classes = 1 #binary classifier (0-1 score after sigmoid)
batch_size = 16


model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={
        'task_name': 'classification',
        'n_channels': predictors,
        'num_class': 2
    },
)
model.init()

criterion = torch.nn.BCEWithLogitsLoss() # loss fn
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

causal_mask = torch.tril(torch.ones((context_length, context_length), dtype=torch.bool))

# training data shape: [batches, preditors, context length (512)]
# labels shape : [batches, ]
train_dataloader = torch.DataLoader(train, batch_size=16, shuffle=True) 

for data, labels in tqdm(train_dataloader, desc="Training"):
    #embed to computer for faster computation?
    data = data.to(model.device)
    labels = labels.to(model.device).float() # Must be float for BCE
    mask = causal_mask.to(model.device)

    # forward [batch_size, n_channels, context length]
    output = model(x_enc=data, attention_mask = mask) # get logits from forward pass
    
    # backward
    logits_for_loss = output.logits.squeeze(-1) #remove class dimension

    loss = criterion(output.logits_for_loss, labels) # compare logits to labels
    optimizer.zero_grad() # reset gradients
    loss.backward() # calculate gradients with backwards pass
    optimizer.step() # update weights
    
    print(f"loss: {loss.item():.3f}")

Below is a from Google timesFM https://github.com/google-research/timesfm
this is a timeseries foundation model - decoder only
This uses causal masking, where on a single pass, calculates
P(x t+1 | x 1:t) for t in input tensor. Much more efficient than training a single y for each pass.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high") #tensor float32 - speeds up matrix performance

#download 200 million parameter model
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")


model.compile(
    timesfm.ForecastConfig(
        max_context=1024, #historic data points to look at
        max_horizon=256, #future datapoints to predict
        normalize_inputs=True, #scale inputs automatically
        use_continuous_quantile_head=True, #output continuous range not single point
        force_flip_invariance=True, #passes with all values as positive, then multiplies by -1 and passes again (supposed to add stability but needs experimenting and research)
        infer_is_positive=False, # all values should be positive (this is false as transaction qty can be negative)
        fix_quantile_crossing=True, # quantiles predicted by different heads are in logical numeric order
    )
)
point_forecast, quantile_forecast = model.forecast(
    horizon=12, # predict 12 steps in the future
    inputs=[
        np.linspace(0, 1, 100),
        np.sin(np.linspace(0, 20, 67)),
    ],
)
point_forecast.shape  # (2, 12)
quantile_forecast.shape  # (2, 12, 10): mean, then 10th to 90th quantiles.


Ok backtrack again. See if I can actually use moment and pass 512 input array to a forecast model with a causal mask and output a 511 vector. If this mask works, the first element of the vector will be trained on the first row of input array, next on first 2 rows, and so on.

In [1]:
import torch

model = torch.load("moment_checkpoint_latest.pt")

In [2]:
from torch.utils.data import DataLoader, Dataset
import numpy as np
from helper import FoundationDataset, embModel

data = np.load(r"C:\Python Projects\ML Transactions\data_cache\obs_2025.npz")
val_dataset = FoundationDataset(data["temporal_val"], data["dense_val"], data["mask_val"], data["Y_val"])

c:\Python Projects\Late Dispatches\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Python Projects\Late Dispatches\venv\Lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [7]:
from tqdm import tqdm

checkpoint = torch.load("moment_checkpoint_latest.pt")
model = embModel()

model.load_state_dict(checkpoint['model_state_dict'])

val_dataloader = DataLoader(val_dataset, batch_size=256, shuffle=True, pin_memory=True)

model.eval() # Set to evaluation mode
val_loss = 0
criterion = torch.nn.BCEWithLogitsLoss() # loss fn

with torch.no_grad(): # Disable gradient calculation
    for val_batch in tqdm(val_dataloader, desc="Training"):
        vX = val_batch['X']
        vmask = val_batch['mask']
        vY = val_batch['Y']
        
        v_out = model(vX, vmask)
        v_loss = criterion(v_out.logits, vY)
        val_loss += v_loss.item()
        
avg_val_loss = val_loss / len(val_dataloader)
print(f"--- Random Sample Validation Loss: {avg_val_loss:.3f} ---")

model.train() # return to training mode

Training:   1%|          | 4/435 [01:20<2:24:52, 20.17s/it]


KeyboardInterrupt: 